In [1]:
# 2.1 a) What Unicode character does chr(0) return?
chr(0)

'\x00'

In [2]:
chr(ord('s')) == 's'

True

In [3]:
# 2.1 b) How does this character’s string representation (__repr__()) differ from its printed representation?
print(chr(0)) # this is the null character (ASCII code 0), not visible when printed
repr(chr(0)) # this will show the string representation of the null character, which is '\x00'
# print(0)
# repr(0)

 


"'\\x00'"

In [4]:
# 2.1 c) What happens when this character occurs in text?
print("this is a test" + chr(0) + "string")
"this is a test" + chr(0) + "string"

this is a test string


'this is a test\x00string'

In [5]:
# 1 byte = 8 bit

In [6]:
# 2.2 

test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")

print(utf8_encoded)


b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'


In [7]:
print(type(utf8_encoded))

<class 'bytes'>


In [8]:
# Get the byte values for the encoded string (integers from 0 to 255).
list(utf8_encoded)

[104,
 101,
 108,
 108,
 111,
 33,
 32,
 227,
 129,
 147,
 227,
 130,
 147,
 227,
 129,
 171,
 227,
 129,
 161,
 227,
 129,
 175,
 33]

In [9]:
# One byte does not necessarily correspond to one Unicode character!
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode("utf-8"))

13
23
hello! こんにちは!


In [10]:
# 2.2 a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various input strings.
# space efficient, compatiable everywhere

In [11]:
# 2.2 b)

In [12]:
encode_niu = "牛🕸️".encode("utf-8")
print(encode_niu)
print([bytes([b]) for b in encode_niu])
print(b'h'.decode('utf-8'))
print(b'\x9b'.decode('utf-8'))


b'\xe7\x89\x9b\xf0\x9f\x95\xb8\xef\xb8\x8f'
[b'\xe7', b'\x89', b'\x9b', b'\xf0', b'\x9f', b'\x95', b'\xb8', b'\xef', b'\xb8', b'\x8f']
h


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x9b in position 0: invalid start byte

In [13]:
encode_niu = "hello".encode("utf-8")
print(encode_niu)
print([bytes([b]) for b in encode_niu])

b'hello'
[b'h', b'e', b'l', b'l', b'o']


In [14]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("牛".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe7 in position 0: unexpected end of data

In [15]:
# one single character can be encoded into multiple bytes, which should be decoded together.

In [16]:
# 2.2 c) Give a two byte sequence that does not decode to any Unicode character(s).
print(b'\x9b \x89'.decode('utf-8'))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x9b in position 0: invalid start byte

In [17]:
# 2.3 

In [18]:
# 2.4 Byte-Pair Encoding Tokenizer

In [ ]:
special_tokens = {"<|endoftext|>"}

def vocab_init(text: str) -> set[str]:
    vocab = set(text)
    vocab.add("<|endoftext|>")
    return vocab

def pretokenize(text: str) -> dict[str, int]:
    import regex as re
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    token_list = re.findall(PAT, text)
    freq = {}
    for i in token_list:
        freq[i] = freq.get(i, 0) + 1
    return freq

def get_byte_freq(token_freq: dict[str, int]) -> dict[tuple, int]:
    byte_freq = {}
    for k, v in token_freq.items():
        byte_freq[tuple(k)] = v
    return byte_freq

def calculate_pair_freq(byte_freq: dict[tuple, int]) -> dict[tuple[str, str], int]:
    pair_freq = {}
    for k, v in byte_freq.items():
        if len(k) < 2:
            continue
        for i in range(1, len(k)):
            pair = (k[i-1], k[i])
            pair_freq[pair] = pair_freq.get(pair, 0) + 1
    return pair_freq

def get_most_freq_pair(pair_freq: dict[tuple[str, str], int]) -> tuple[str, str]:
    return max(pair_freq.items(), key=lambda x: x[1])[0]

def merge_pair(byte_freq: dict[tuple[str, str], int], pair_to_merge: tuple[str, str]) -> dict[tuple[str, str], int]:
    merged_freq = {}
    for k, v in byte_freq.items():
        i = 0
        k_new = []
        while i < len(k):
            if k[i:i+len(pair_to_merge)] == pair_to_merge:
                # then update k with merged byte.
                k_new.append(''.join(pair_to_merge))
                i += len(pair_to_merge)
            else:
                k_new.append(k[i])
                i += 1
        # byte_freq[tuple(k_new)] = v
        # del byte_freq[k]
        merged_freq[tuple(k_new)] = v
    return merged_freq





In [2]:
def train_bpe_test(num_merges: int = 3) -> tuple[dict[str, int], dict[tuple, int]]:
    text = '''
    This year, we saw a dazzling application of machine learning.
    The OpenAI GPT-2 exhibited impressive ability of writing coherent and passionate essays
    that exceed what we anticipated current language models are able to produce. The GPT-2 wasn’t a
    particularly novel architecture – it’s architecture is very similar to the decoder-only transformer.
    The GPT2 was, however, a very large, transformer-based language model trained on a massive dataset.
    In this post, we’ll look at the architecture that enabled the model to produce its results.
    We will go into the depths of its self-attention layer. And then we’ll look at applications for the
    decoder-only transformer beyond language modeling.
    My goal here is to also supplement my earlier post, The Illustrated Transformer, with more visuals
    explaining the inner-workings of transformers, and how they’ve evolved since the original paper.
    My hope is that this visual language will hopefully make it easier to explain later Transformer-based
    models as their inner-workings continue to evolve.
    '''

    vocab = vocab_init(text)
    # idx = max([ord(c) for c in vocab]) + 1
    idx = 256
    # print(vocab_init(text))
    # print(idx)
    tokens_freq = pretokenize(text)
    # print(tokens_freq)
    byte_freq = get_byte_freq(tokens_freq)
    # print(byte_freq)

    # token_list = {k : ord(k) for k in vocab}
    token_list = {k : k.encode('utf-8') for k in vocab}

    for _ in range(num_merges):
        pair_freq = calculate_pair_freq(byte_freq)
        if not pair_freq:
            break

        pair_to_merge = get_most_freq_pair(pair_freq)
        print("merge:", pair_to_merge)

        byte_freq = merge_pair(byte_freq, pair_to_merge)
        token_list[''.join(pair_to_merge)] = idx
        idx += 1

    # print(token_list)
    # print(byte_freq)
    return token_list, byte_freq

if __name__ == "__main__":
    token_list, byte_freq = train_bpe_test()
    print('current token list:', token_list)
    print('current byte freq:', byte_freq)

merge: ('i', 'n')
merge: ('e', 'r')
merge: (' ', 'a')
current token list: {'’': b'\xe2\x80\x99', 'o': b'o', 'O': b'O', '-': b'-', 'c': b'c', 'M': b'M', 'm': b'm', 'P': b'P', 'f': b'f', 'h': b'h', 'd': b'd', 'w': b'w', 'A': b'A', ',': b',', 'G': b'G', 'r': b'r', '2': b'2', '\n': b'\n', 't': b't', 'i': b'i', 'z': b'z', 'I': b'I', '–': b'\xe2\x80\x93', 'l': b'l', '.': b'.', 'b': b'b', 'T': b'T', 'v': b'v', 'p': b'p', 'W': b'W', 'a': b'a', 'x': b'x', 's': b's', 'g': b'g', 'k': b'k', 'y': b'y', ' ': b' ', 'n': b'n', '<|endoftext|>': b'<|endoftext|>', 'e': b'e', 'u': b'u', 'in': 256, 'er': 257, ' a': 258}
current byte freq: {('\n', ' ', ' ', ' '): 12, (' ', 'T', 'h', 'i', 's'): 1, (' ', 'y', 'e', 'a', 'r'): 1, (',',): 8, (' ', 'w', 'e'): 4, (' ', 's', 'a', 'w'): 1, (' a',): 4, (' ', 'd', 'a', 'z', 'z', 'l', 'in', 'g'): 1, (' a', 'p', 'p', 'l', 'i', 'c', 'a', 't', 'i', 'o', 'n'): 1, (' ', 'o', 'f'): 4, (' ', 'm', 'a', 'c', 'h', 'in', 'e'): 1, (' ', 'l', 'e', 'a', 'r', 'n', 'in', 'g'): 1, ('.'

In [6]:
def train_bpe(input_path: str, vocab_size: int, special_tokens: list[str]) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    data = open(input_path, "r").read()
    # remove special tokens from data
    for token in special_tokens:
        data = data.replace(token, "")

    vocab = vocab_init(data)
    token_freq = pretokenize(data)
    byte_freq = get_byte_freq(token_freq)
    token_list = {int(k.encode('utf-8')[0]) : k.encode('utf-8') for k in vocab}
    idx = 256
    
    merges = []

    while len(token_list) < vocab_size:
        pair_freq = calculate_pair_freq(byte_freq)
        if not pair_freq:
            break

        pair_to_merge = get_most_freq_pair(pair_freq)
        pair_to_merge_bytes = (pair_to_merge[0].encode('utf-8'), pair_to_merge[1].encode('utf-8'))
        merges.append(pair_to_merge_bytes)

        byte_freq = merge_pair(byte_freq, pair_to_merge)
        # token_list[''.join(pair_to_merge)] = idx
        token_list[idx] = ''.join(pair_to_merge).encode('utf-8')
        idx += 1

    return token_list, merges

if __name__ == "__main__":
    token_list, merges = train_bpe("/Users/ruoxinwang/Desktop/PhD/CS336/cs336-assignment1-basics/data/test_mini.txt", 30000, list(special_tokens))
    print('current token list:', token_list)
    print('current merges:', merges)

current token list: {111: b'o', 79: b'O', 70: b'F', 75: b'K', 45: b'-', 99: b'c', 106: b'j', 77: b'M', 109: b'm', 102: b'f', 104: b'h', 119: b'w', 100: b'd', 63: b'?', 65: b'A', 44: b',', 33: b'!', 114: b'r', 10: b'\n', 116: b't', 105: b'i', 226: b'\xe2\x80\x9d', 122: b'z', 73: b'I', 108: b'l', 46: b'.', 98: b'b', 72: b'H', 67: b'C', 89: b'Y', 34: b'"', 58: b':', 76: b'L', 69: b'E', 68: b'D', 84: b'T', 118: b'v', 112: b'p', 87: b'W', 97: b'a', 120: b'x', 115: b's', 103: b'g', 107: b'k', 78: b'N', 39: b"'", 121: b'y', 83: b'S', 110: b'n', 32: b' ', 66: b'B', 60: b'<|endoftext|>', 101: b'e', 117: b'u', 256: b'ed', 257: b' s', 258: b'in', 259: b'er', 260: b' t', 261: b' b', 262: b'ar', 263: b' c', 264: b' f', 265: b' h', 266: b' w', 267: b' l', 268: b'ou', 269: b'ing', 270: b'at', 271: b' d', 272: b'on', 273: b'or', 274: b'en', 275: b'an', 276: b' p', 277: b' r', 278: b'll', 279: b'ow', 280: b'me', 281: b' g', 282: b' a', 283: b'le', 284: b'is', 285: b' th', 286: b' n', 287: b' e', 288: b

In [5]:
t = "test, @@@*()<|end_of_text|>, test, test, today is a good day."
# print(vocab_init(t))
# print([i.encode("utf-8") for i in vocab_init(t)])
# print([ord(i) for i in vocab_init(t)])

# print(pretokenize(t))
token_freq = pretokenize(t)
byte_freq = get_byte_freq(token_freq)
print(byte_freq)
pair_freq = calculate_pair_freq(byte_freq)
print(pair_freq)
pair_to_merge = get_most_freq_pair(pair_freq)
merged_freq = merge_pair(byte_freq, pair_to_merge)


{('t', 'e', 's', 't'): 1, (',',): 3, (' ', '@', '@', '@', '*', '(', ')', '<', '|'): 1, ('e', 'n', 'd'): 1, ('_',): 2, ('o', 'f'): 1, ('t', 'e', 'x', 't'): 1, ('|', '>', ','): 1, (' ', 't', 'e', 's', 't'): 2, (' ', 't', 'o', 'd', 'a', 'y'): 1, (' ', 'i', 's'): 1, (' ', 'a'): 1, (' ', 'g', 'o', 'o', 'd'): 1, (' ', 'd', 'a', 'y'): 1, ('.',): 1}
{('t', 'e'): 3, ('e', 's'): 2, ('s', 't'): 2, (' ', '@'): 1, ('@', '@'): 2, ('@', '*'): 1, ('*', '('): 1, ('(', ')'): 1, (')', '<'): 1, ('<', '|'): 1, ('e', 'n'): 1, ('n', 'd'): 1, ('o', 'f'): 1, ('e', 'x'): 1, ('x', 't'): 1, ('|', '>'): 1, ('>', ','): 1, (' ', 't'): 2, ('t', 'o'): 1, ('o', 'd'): 2, ('d', 'a'): 2, ('a', 'y'): 2, (' ', 'i'): 1, ('i', 's'): 1, (' ', 'a'): 1, (' ', 'g'): 1, ('g', 'o'): 1, ('o', 'o'): 1, (' ', 'd'): 1}


In [ ]:
"a".encode("utf-8")
int("a".encode("utf-8")[0])

b'a'

In [ ]:
print(len(token_list))  # how many entries before any merges?
print(256 in token_list)  # sanity check on indexing
# more importantly:
print(len([k for k in token_list if k < 256]))  # how many base bytes do you actually have?

In [ ]:
# 2.5

In [5]:
# def vocab_init(text: str) -> set[str]:
#     vocab = set(text)
#     # vocab.add("<|endoftext|>")
#     return vocab

def pretokenize(text: str) -> dict[str, int]:
    import regex as re
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    token_list = re.findall(PAT, text)
    freq = {}
    for i in token_list:
        freq[i] = freq.get(i, 0) + 1
    return freq

def get_byte_freq(token_freq: dict[str, int]) -> dict[tuple, int]:
    byte_freq = {}
    for k, v in token_freq.items():
        # byte_freq[tuple(k.encode('utf-8'))] = v
        byte_freq[tuple(k)] = v
    return byte_freq

def calculate_pair_freq(byte_freq: dict[tuple, int]) -> dict[tuple[str, str], int]:
    pair_freq = {}
    for k, v in byte_freq.items():
        if len(k) < 2:
            continue
        for i in range(1, len(k)):
            pair = (k[i-1], k[i])
            pair_freq[pair] = pair_freq.get(pair, 0) + v
    return pair_freq

def get_most_freq_pair(pair_freq: dict[tuple[str, str], int]) -> tuple[str, str]:
    # return max(pair_freq.items(), key=lambda x: x[1])[0]
    return max(pair_freq.items(), key=lambda x: (x[1], x[0]))[0]

def merge_pair(byte_freq: dict[tuple[str, str], int], pair_to_merge: tuple[str, str]) -> dict[tuple[str, str], int]:
    merged_freq = {}
    for k, v in byte_freq.items():
        i = 0
        k_new = []
        while i < len(k):
            if k[i:i+len(pair_to_merge)] == pair_to_merge:
                # then update k with merged byte.
                # k_new.append(''.join(pair_to_merge))
                k_new.append(''.join(chr(b) for b in pair_to_merge))
                i += len(pair_to_merge)
            else:
                k_new.append(k[i])
                i += 1
        # byte_freq[tuple(k_new)] = v
        # del byte_freq[k]
        merged_freq[tuple(k_new)] = v
    return merged_freq


def train_bpe(input_path: str, vocab_size: int, special_tokens: list[str]) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    data = open(input_path, "r").read()
    # remove special tokens from data
    for token in special_tokens:
        data = data.replace(token, "")

    # vocab = vocab_init(data)
    token_freq = pretokenize(data)
    byte_freq = get_byte_freq(token_freq)
    token_list = {i: bytes([i]) for i in range(256)}
    idx = 256
    # idx = len(token_list)
    # token_list = {int(k.encode('utf-8')[0]) : k.encode('utf-8') for k in vocab}
    
    merges = []

    while len(token_list) + len(special_tokens) < vocab_size:
        pair_freq = calculate_pair_freq(byte_freq)
        if not pair_freq:
            break

        pair_to_merge = get_most_freq_pair(pair_freq)
        pair_to_merge_bytes = (pair_to_merge[0].encode('utf-8'), pair_to_merge[1].encode('utf-8'))
        merges.append(pair_to_merge_bytes)

        byte_freq = merge_pair(byte_freq, pair_to_merge)
        # token_list[''.join(pair_to_merge)] = idx
        token_list[idx] = ''.join(pair_to_merge).encode('utf-8')
        idx += 1

    # add special tokens after merges
    for token in special_tokens:
        token_list[idx] = token.encode('utf-8')
        idx += 1

    return token_list, merges


In [7]:
# mini test
t = "lower lower lowest, newer"
# print(vocab_init(t))
# print([i.encode("utf-8") for i in vocab_init(t)])
# print([ord(i) for i in vocab_init(t)])

# print(pretokenize(t))
token_freq = pretokenize(t)
byte_freq = get_byte_freq(token_freq)
print(byte_freq)
pair_freq = calculate_pair_freq(byte_freq)
print(pair_freq)
pair_to_merge = get_most_freq_pair(pair_freq)
merged_freq = merge_pair(byte_freq, pair_to_merge)
print(merged_freq)

{('l', 'o', 'w', 'e', 'r'): 1, (' ', 'l', 'o', 'w', 'e', 'r'): 1, (' ', 'l', 'o', 'w', 'e', 's', 't'): 1, (',',): 1, (' ', 'n', 'e', 'w', 'e', 'r'): 1}
{('l', 'o'): 3, ('o', 'w'): 3, ('w', 'e'): 4, ('e', 'r'): 3, (' ', 'l'): 2, ('e', 's'): 1, ('s', 't'): 1, (' ', 'n'): 1, ('n', 'e'): 1, ('e', 'w'): 1}


TypeError: 'str' object cannot be interpreted as an integer

In [7]:
! uv run pytest tests/test_train_bpe.py

============================= test session starts ==============================
platform darwin -- Python 3.11.8, pytest-8.4.1, pluggy-1.6.0
rootdir: /Users/ruoxinwang/Desktop/PhD/CS336/cs336-assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.2, anyio-4.13.0
collected 0 items                                                              

============================ no tests ran in 0.00s =============================
ERROR: file or directory not found: tests/test_train_bpe.py



In [1]:
1776047360.575041 - 1776047358.861283

1.7137579917907715

In [ ]:
import regex as re
import heapq
from collections import defaultdict


PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

# def pretokenize(text: str) -> dict[str, int]:
#     freq = {}
#     # token_list = re.findall(PAT, text)
#     token_list = [m.group(0) for m in re.finditer(PAT, text)]   
#     for i in token_list:
#         freq[i] = freq.get(i, 0) + 1
#     return freq

def build_token_freq(data: str, special_tokens: list[str]) -> dict[str, int]:
    '''
    Build a frequency dictionary for tokens in the given data.
    {} -> {'hello': 5, 'world': 3, ...}
    '''
    token_freq = {}

    if special_tokens:
        split_pat = "|".join(re.escape(t) for t in special_tokens)
        chunks = re.split(split_pat, data)
    else:
        chunks = [data]

    for chunk in chunks:
        if not chunk:
            continue
        for m in re.finditer(PAT, chunk):
            tok = m.group(0)
            token_freq[tok] = token_freq.get(tok, 0) + 1

    return token_freq

def get_byte_freq(token_freq: dict[str, int]) -> dict[tuple, int]:
    '''
    Convert token frequencies to byte frequencies.
    {} -> {(b'h', b'e', b'l', b'l', b'o'): 5, ...}
    '''
    byte_freq = {}
    for k, v in token_freq.items():
        # byte_freq[tuple(k)] = v
        byte_freq[tuple(bytes([b]) for b in k.encode("utf-8"))] = v
    return byte_freq

def calculate_pair_freq(byte_freq: dict[tuple, int]) -> dict[tuple[str, str], int]:
    '''
    Calculate the frequency of adjacent byte pairs in the byte frequency dictionary.
    {(b'h', b'e', b'l', b'l', b'o'): 5
    '''
    pair_freq = {}
    for k, v in byte_freq.items():
        if len(k) < 2:
            continue
        for i in range(1, len(k)):
            pair = (k[i-1], k[i])
            pair_freq[pair] = pair_freq.get(pair, 0) + v
    return pair_freq

# def calculate_pair_idx(token_freq: dict[str, int], pair_freq: dict[tuple[str, str], int]) -> dict[tuple[str, str], list[tuple]]:
#     pair_idx = {}
#     for k, _ in pair_freq.items():
#         p = (k[0] + k[1]).decode("utf-8")
#         pair_idx[k] = [tuple(token) for token in token_freq.keys() if p in token]
#     return pair_idx

def calculate_pair_idx(byte_freq: dict[tuple, int]) -> dict[tuple, set[tuple]]:
    pair_idx = {}
    for seq in byte_freq:
        for i in range(len(seq) - 1):
            p = (seq[i], seq[i+1])
            pair_idx.setdefault(p, set()).add(seq)
    return pair_idx

def get_most_freq_pair(pair_freq: dict[tuple[str, str], int]) -> tuple[str, str]:
    # return max(pair_freq.items(), key=lambda x: x[1])[0]
    return max(pair_freq.items(), key=lambda x: (x[1], x[0]))[0]

def merge_pair(byte_freq: dict[tuple[str, str], int], pair_to_merge: tuple[str, str]) -> dict[tuple[str, str], int]:
    merged_freq = {}
    for k, v in byte_freq.items():
        i = 0
        k_new = []
        while i < len(k):
            if i + 1 < len(k) and k[i] == pair_to_merge[0] and k[i+1] == pair_to_merge[1]:
                # k_new.append(''.join(pair_to_merge))
                k_new.append(k[i] + k[i+1])
                i += 2
            else:
                k_new.append(k[i])
                i += 1

        new_k = tuple(k_new)
        merged_freq[new_k] = merged_freq.get(new_k, 0) + v

    return merged_freq

def update_pair_freq(pair_freq: dict[tuple[str, str], int], byte_freq: dict[tuple[str, str], int], pair_to_merge: tuple[str, str]) -> dict[tuple[str, str], int]:
    updated_pair_freq = {}
    for k, v in pair_freq.items():
        if pair_to_merge in k:
            continue
        updated_pair_freq[k] = v

    for k, v in byte_freq.items():
        if len(k) < 2:
            continue
        for i in range(1, len(k)):
            pair = (k[i-1], k[i])
            if pair == pair_to_merge:
                new_pair = (k[i-1] + k[i],)
                updated_pair_freq[new_pair] = updated_pair_freq.get(new_pair, 0) + v
            else:
                updated_pair_freq[pair] = updated_pair_freq.get(pair, 0) + v

    return updated_pair_freq

def train_bpe(input_path, vocab_size, special_tokens):
    data = open(input_path).read()
    token_freq = build_token_freq(data, special_tokens)

    token_list = {i: bytes([i]) for i in range(256)}
    idx = 256
    for token in special_tokens:
        token_list[idx] = token.encode('utf-8')
        idx += 1

    byte_freq = get_byte_freq(token_freq)

    # Build pair_freq AND pair_idx in ONE pass (replaces both calculate_* calls)
    pair_freq = defaultdict(int)
    pair_idx  = defaultdict(set)
    for seq, freq in byte_freq.items():
        for i in range(len(seq) - 1):
            p = (seq[i], seq[i+1])
            pair_freq[p] += freq
            pair_idx[p].add(seq)

    # Heap for O(log n) max lookup instead of O(n) max() each iteration
    # (-freq, pair) because heapq is a min-heap
    heap = [(-freq, pair) for pair, freq in pair_freq.items()]
    heapq.heapify(heap)

    merges = []

    while idx < vocab_size:
        # Lazy deletion: skip heap entries that are stale
        while heap:
            neg_freq, pair = heap[0]
            if pair_freq.get(pair, 0) == -neg_freq:
                break
            heapq.heappop(heap)   # stale, discard

        if not heap:
            break

        neg_freq, pair_to_merge = heapq.heappop(heap)
        A, B = pair_to_merge
        AB = A + B

        affected_seqs = list(pair_idx.pop(pair_to_merge, set()))
        del pair_freq[pair_to_merge]

        for seq in affected_seqs:
            freq = byte_freq.pop(seq)

            # Step 1: undo old contributions
            for i in range(len(seq) - 1):
                p = (seq[i], seq[i+1])
                if p == pair_to_merge:
                    continue
                pair_freq[p] -= freq
                if pair_freq[p] <= 0:
                    del pair_freq[p]
                pair_idx[p].discard(seq)

            # Step 2: merge
            new_seq = []
            i = 0
            while i < len(seq):
                if i+1 < len(seq) and seq[i] == A and seq[i+1] == B:
                    new_seq.append(AB)
                    i += 2
                else:
                    new_seq.append(seq[i])
                    i += 1
            new_seq = tuple(new_seq)

            # Step 3: add new contributions and push to heap
            byte_freq[new_seq] = byte_freq.get(new_seq, 0) + freq
            for i in range(len(new_seq) - 1):
                p = (new_seq[i], new_seq[i+1])
                pair_freq[p] = pair_freq.get(p, 0) + freq
                pair_idx[p].add(new_seq)
                heapq.heappush(heap, (-pair_freq[p], p))  # push updated entry; old one becomes stale

        merges.append(pair_to_merge)
        token_list[idx] = AB
        idx += 1

    return token_list, merges

# def train_bpe(input_path: str, vocab_size: int, special_tokens: list[str]) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
#     data = open(input_path, "r").read()

#     # # separate data into chunks based on <|endoftext|>
#     # chunks = data.split("<|endoftext|>")

#     # token_freq = {}
#     # # pretokenize each chunk and accumulate token frequencies
#     # for chunk in chunks:
#     #     chunk_freq = pretokenize(chunk)
#     #     for k, v in chunk_freq.items():
#     #         token_freq[k] = token_freq.get(k, 0) + v

#     token_freq = build_token_freq(data, special_tokens)

#     # initialize token list with single byte tokens
#     # token_list = {int(k.encode('utf-8')[0]) : k.encode('utf-8') for k in vocab}
#     token_list = {i: bytes([i]) for i in range(256)}
#     idx = 256

#     # remove special tokens from data
#     for token in special_tokens:
#         token_list[idx] = token.encode('utf-8')
#         # data = data.replace(token, "")
#         idx += 1

#     byte_freq = get_byte_freq(token_freq)
#     pair_freq = calculate_pair_freq(byte_freq)
    
#     # add a reverse index for byte pairs to bytes
#     pair_idx = calculate_pair_idx(token_freq, pair_freq)

#     # store the merged pair and tokens that include the pair,
#     merges = []

#     while idx < vocab_size:
#         pair_to_merge = get_most_freq_pair(pair_freq)
#         merges.append(pair_to_merge)  # already bytes!

#         A, B = pair_to_merge
#         AB = A + B

#         # --- only look at affected sequences via pair_idx ---
#         affected_seqs = list(pair_idx.pop(pair_to_merge, []))
#         del pair_freq[pair_to_merge]

#         for seq in affected_seqs:
#             freq = byte_freq.pop(seq)

#             # Step 1: undo this seq's contributions to pair_freq / pair_idx
#             for i in range(len(seq) - 1):
#                 p = (seq[i], seq[i+1])
#                 if p == pair_to_merge:
#                     continue          # already removed above
#                 pair_freq[p] -= freq
#                 if pair_freq[p] <= 0:
#                     del pair_freq[p]
#                 if p in pair_idx:
#                     pair_idx[p].discard(seq)

#             # Step 2: build the merged sequence
#             new_seq = []
#             i = 0
#             while i < len(seq):
#                 if i+1 < len(seq) and seq[i] == A and seq[i+1] == B:
#                     new_seq.append(AB)
#                     i += 2
#                 else:
#                     new_seq.append(seq[i])
#                     i += 1
#             new_seq = tuple(new_seq)

#             # Step 3: add new_seq's contributions back
#             byte_freq[new_seq] = byte_freq.get(new_seq, 0) + freq
#             for i in range(len(new_seq) - 1):
#                 p = (new_seq[i], new_seq[i+1])
#                 pair_freq[p] = pair_freq.get(p, 0) + freq
#                 pair_idx.setdefault(p, set()).add(new_seq)
        
#         # byte_freq = merge_pair(byte_freq, pair_to_merge)
#         # pair_freq = calculate_pair_freq(byte_freq)
        
#         # value as bytes, key as int
#         # token_list[idx] = ''.join(pair_to_merge)
#         # token_list[idx] = pair_to_merge[0] + pair_to_merge[1]
#         token_list[idx] = AB
#         idx += 1

    # return token_list, merges

In [30]:
data = '''
low low low low low
lower lower widest widest widest
newest newest newest newest newest newest
'''

special_tokens = ["<|endoftext|>"]

In [31]:
token_freq = build_token_freq(data, special_tokens)
print(token_freq)

{'\n': 4, 'low': 1, ' low': 4, 'lower': 1, ' lower': 1, ' widest': 3, 'newest': 1, ' newest': 5}


In [33]:
token_list = {i: bytes([i]) for i in range(256)}
print(token_list)
print([i for i in token_list if i < 256])

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

In [34]:
byte_freq = get_byte_freq(token_freq)
print(byte_freq)

{(b'\n',): 4, (b'l', b'o', b'w'): 1, (b' ', b'l', b'o', b'w'): 4, (b'l', b'o', b'w', b'e', b'r'): 1, (b' ', b'l', b'o', b'w', b'e', b'r'): 1, (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3, (b'n', b'e', b'w', b'e', b's', b't'): 1, (b' ', b'n', b'e', b'w', b'e', b's', b't'): 5}


In [35]:
pair_freq = calculate_pair_freq(byte_freq)
print(pair_freq)

{(b'l', b'o'): 7, (b'o', b'w'): 7, (b' ', b'l'): 5, (b'w', b'e'): 8, (b'e', b'r'): 2, (b' ', b'w'): 3, (b'w', b'i'): 3, (b'i', b'd'): 3, (b'd', b'e'): 3, (b'e', b's'): 9, (b's', b't'): 9, (b'n', b'e'): 6, (b'e', b'w'): 6, (b' ', b'n'): 5}


In [37]:
# pair_words = get_pair_words(token_freq, pair_freq)
# print(pair_words)
pair_idx = calculate_pair_idx(token_freq, pair_freq)
print(pair_idx)

{(b'l', b'o'): [('l', 'o', 'w'), (' ', 'l', 'o', 'w'), ('l', 'o', 'w', 'e', 'r'), (' ', 'l', 'o', 'w', 'e', 'r')], (b'o', b'w'): [('l', 'o', 'w'), (' ', 'l', 'o', 'w'), ('l', 'o', 'w', 'e', 'r'), (' ', 'l', 'o', 'w', 'e', 'r')], (b' ', b'l'): [(' ', 'l', 'o', 'w'), (' ', 'l', 'o', 'w', 'e', 'r')], (b'w', b'e'): [('l', 'o', 'w', 'e', 'r'), (' ', 'l', 'o', 'w', 'e', 'r'), ('n', 'e', 'w', 'e', 's', 't'), (' ', 'n', 'e', 'w', 'e', 's', 't')], (b'e', b'r'): [('l', 'o', 'w', 'e', 'r'), (' ', 'l', 'o', 'w', 'e', 'r')], (b' ', b'w'): [(' ', 'w', 'i', 'd', 'e', 's', 't')], (b'w', b'i'): [(' ', 'w', 'i', 'd', 'e', 's', 't')], (b'i', b'd'): [(' ', 'w', 'i', 'd', 'e', 's', 't')], (b'd', b'e'): [(' ', 'w', 'i', 'd', 'e', 's', 't')], (b'e', b's'): [(' ', 'w', 'i', 'd', 'e', 's', 't'), ('n', 'e', 'w', 'e', 's', 't'), (' ', 'n', 'e', 'w', 'e', 's', 't')], (b's', b't'): [(' ', 'w', 'i', 'd', 'e', 's', 't'), ('n', 'e', 'w', 'e', 's', 't'), (' ', 'n', 'e', 'w', 'e', 's', 't')], (b'n', b'e'): [('n', 'e', 

In [39]:
1778143021.9002318 - 1778143020.206729

1.6935029029846191

In [1]:
from collections.abc import Iterable, Iterator
import ast
import regex as re

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

class BPE_Tokenizer:
    def __init__(self, vocab, merges, special_tokens=None):
        vocab = vocab.copy()
        merges = merges.copy()
        special_tokens = special_tokens or []

    @classmethod
    def from_files(cls, vocab_filepath, merges_filepath, special_tokens=None):
        vocab = {}
        # Parse the whole file so raw control-byte tokens (e.g. '\n', '\t')
        # are handled correctly.
        with open(vocab_filepath, "r", encoding="utf-8", newline="") as f:
            content = f.read()
        for m in re.finditer(r"(?s)(\d+)\t(.*?)(?=\n\d+\t|\Z)", content):
            idx = int(m.group(1))
            token = m.group(2).encode("utf-8", errors="replace")
            vocab[idx] = token
        
        merges = []
        with open(merges_filepath, "r", encoding="utf-8") as f:
            next(f)  # skip header
            next(f)  # skip separator
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) < 4:
                    continue
                a_str, b_str = parts[1].strip(), parts[2].strip()
                a = ast.literal_eval(a_str).encode("utf-8", errors="replace")
                b = ast.literal_eval(b_str).encode("utf-8", errors="replace")
                merges.append((a, b))
        
        return cls(vocab, merges, special_tokens)

    def pre_tokenize(self, text: str, special_tokens=None) -> list[bytes]:
        tokens = []

        if special_tokens:
            split_pat = "|".join(re.escape(t) for t in special_tokens)
            chunks = re.split(split_pat, text)
        else:
            chunks = [text]

        for chunk in chunks:
            if not chunk:
                continue
            for m in re.finditer(PAT, chunk):
                tok = m.group(0)
                tokens.append(tok.encode("utf-8", errors="replace"))

        return tokens
    
    def bpe_merge(byte_list: list[tuple[bytes]], merges: list[tuple[bytes, bytes]]) -> list[tuple[bytes]]:
        # create a mapping of byte pairs to their merged form
        merge_dict = {pair: AB for AB, pair in merges}

        # repeatedly merge byte pairs until no more merges are possible
        while True:
            new_byte_list = []
            i = 0
            while i < len(byte_list):
                if i < len(byte_list) - 1:
                    pair = (byte_list[i], byte_list[i+1])
                    if pair in merge_dict:
                        new_byte_list.append(merge_dict[pair])
                        i += 2  # skip the next byte since it's merged
                        continue
                new_byte_list.append(byte_list[i])
                i += 1
            
            if new_byte_list == byte_list:  # no more merges possible
                break
            byte_list = new_byte_list
        
        return byte_list
    
    def get_ids_from_bytes(self, byte_list: list[tuple[bytes]]) -> list[int]:
        # create a mapping of byte sequences to their token ids
        byte_to_id = {v: k for k, v in self.vocab.items()}

        token_ids = []
        for byte_seq in byte_list:
            if byte_seq in byte_to_id:
                token_ids.append(byte_to_id[byte_seq])
            else:
                raise ValueError(f"Byte sequence {byte_seq} not found in vocabulary")
        
        return token_ids
    
    def encode(self, text: str) -> list[int]:
        # pre-tokenize the text into bytes
        tokens = self.pre_tokenize(text)

        # split tokens into bytes and merge according to merges
        byte_list = [tuple(bytes([t]) for t in tokens.encode("utf-8"))]

        # apply bpe merges
        byte_merges = self.bpe_merge(byte_list, self.merges)

        # convert merged byte sequences to token ids
        token_ids = self.get_ids_from_bytes(byte_merges)

        return token_ids

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        pass

    def decode(self, ids: list[int]) -> str:
        pass


In [1]:
from cs336_basics.section2.tokenizer import BPE_Tokenizer

tokenizer = BPE_Tokenizer.from_files(
    "results/tinystories/vocab.txt", 
    "results/tinystories/merges.txt", 
    special_tokens=["<|endoftext|>"]
)

encoded = tokenizer.encode("Hello, world!")
print(encoded)  

[1028, 2372, 44, 1569, 33]


(a) On 10 sampled TinyStories documents, the TinyStories tokenizer achieved X bytes/token; on 10 sampled OpenWebText documents, the OpenWebText tokenizer achieved Y bytes/token. This means each token represents about X or Y UTF-8 bytes on average.

(b) When I tokenize the OpenWebText sample with the TinyStories tokenizer, the compression ratio drops to Z bytes/token. This happens because the TinyStories tokenizer was trained on simpler story-like text and therefore splits web text, rare words, URLs, and punctuation-heavy content into smaller pieces.

(c) My tokenizer processes approximately T bytes/second. At this rate, tokenizing the 825GB Pile dataset would take about H hours, or D days, assuming similar input and no parallelization.

(d) `uint16` is appropriate because it stores values from 0 to 65,535, which covers both the 10K and 32K vocabularies. It also uses less storage than `int32` or `int64`, making the encoded token arrays more memory- and disk-efficient.